# Phase 2: Cross-Platform Market Matching

## Objective
Match Polymarket events to Kalshi events using semantic similarity.

## What This Does
- Fetches 2,000 Polymarket events and 4,897 Kalshi events
- Computes semantic similarity using sentence-transformers
- Identifies matched market pairs
- Selects top 6 matches for analysis

## Expected Output
`data/processed/matched_markets.json` - 6 market pairs with similarity scores

In [ ]:
# Install required packages
!pip install -q requests pandas numpy sentence-transformers

In [ ]:
import json
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from pathlib import Path
import requests

print('Loading sentence transformer...')
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Ready!')

In [ ]:
# Fetch Polymarket events
print('Fetching Polymarket events...')
poly_events = []
offset = 0
while len(poly_events) < 2000:
    try:
        resp = requests.get(
            'https://gamma-api.polymarket.com/events',
            params={'limit': 100, 'offset': offset},
            timeout=10
        )
        resp.raise_for_status()
        batch = resp.json()
        if not batch:
            break
        poly_events.extend(batch)
        offset += 100
        print(f'  {len(poly_events)} events loaded')
    except Exception as e:
        print(f'Error: {e}')
        break

print(f'Total Polymarket events: {len(poly_events)}')

In [ ]:
# Fetch Kalshi events
print('Fetching Kalshi events...')
kalshi_events = []
cursor = None
while True:
    try:
        params = {'limit': 200}
        if cursor:
            params['cursor'] = cursor
        resp = requests.get(
            'https://api.elections.kalshi.com/trade-api/v2/events',
            params=params,
            timeout=10
        )
        resp.raise_for_status()
        data = resp.json()
        batch = data.get('events', [])
        if not batch:
            break
        kalshi_events.extend(batch)
        cursor = data.get('cursor')
        print(f'  {len(kalshi_events)} events loaded')
        if not cursor:
            break
    except Exception as e:
        print(f'Error: {e}')
        break

print(f'Total Kalshi events: {len(kalshi_events)}')

In [ ]:
# Extract titles and encode
print('Encoding event titles...')
poly_titles = [e.get('title', '') for e in poly_events]
kalshi_titles = [e.get('event_ticker', '') + ': ' + e.get('title', '') for e in kalshi_events]

poly_embeddings = model.encode(poly_titles, show_progress_bar=True)
kalshi_embeddings = model.encode(kalshi_titles, show_progress_bar=True)

print(f'Polymarket embeddings: {poly_embeddings.shape}')
print(f'Kalshi embeddings: {kalshi_embeddings.shape}')

In [ ]:
# Compute similarity
print('Computing similarity matrix...')
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(poly_embeddings, kalshi_embeddings)
print(f'Similarity matrix shape: {similarity_matrix.shape}')

# Find matches
matches = []
for i, poly_event in enumerate(poly_events):
    best_idx = np.argmax(similarity_matrix[i])
    best_score = similarity_matrix[i][best_idx]
    
    if best_score >= 0.75:  # threshold
        kalshi_event = kalshi_events[best_idx]
        matches.append({
            'question': poly_event.get('title'),
            'poly_slug': poly_event.get('slug'),
            'poly_event_id': poly_event.get('id'),
            'kalshi_event_ticker': kalshi_event.get('event_ticker'),
            'similarity': float(best_score),
            'poly_volume_usd': poly_event.get('volume_usd'),
            'poly_num_markets': len(poly_event.get('markets', []))
        })

# Sort by similarity
matches = sorted(matches, key=lambda x: x['similarity'], reverse=True)

print(f'Found {len(matches)} matches at threshold >= 0.75')
print(f'\nTop 10 matches:')
for match in matches[:10]:
    print(f"  {match['similarity']:.3f} - {match['question'][:50]}...")

In [ ]:
# Select top 6 markets (with sports for whale detection)
selected = matches[:6]

print('Selected 6 market pairs:')
for i, m in enumerate(selected, 1):
    print(f"\n{i}. {m['question']}")
    print(f"   Similarity: {m['similarity']:.3f}")
    print(f"   Polymarket volume: ${m['poly_volume_usd']:,.0f}")
    print(f"   Polymarket markets: {m['poly_num_markets']}")

In [ ]:
# Save matched markets
import os
os.makedirs('data/processed', exist_ok=True)

with open('data/processed/matched_markets.json', 'w') as f:
    json.dump(selected, f, indent=2)

print('✅ Saved: data/processed/matched_markets.json')